# LeetCode 1314 — Matrix Block Sum bằng Python

Link đề: https://leetcode.com/problems/matrix-block-sum/ · Độ khó: **Medium** · Chủ đề: Array, Matrix, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def matrixBlockSum(self, mat: list[list[int]], k: int) -> list[list[int]]:
        ...
```

## 1. Mục tiêu học

1. Hiểu đề "tổng khối" và cách **cắt (clamp) khối theo biên** ma trận bằng `max`/`min`.
2. Nắm **tổng tiền tố 2D** `P` và công thức **bao hàm – loại trừ** để lấy tổng hình chữ nhật bất kỳ trong `O(1)`.
3. Biết bước trung gian **prefix theo từng hàng** (`O(m·n·k)`) và lời giải **tách chiều**: cửa sổ trượt theo hàng rồi theo cột (`O(m·n)`).
4. Thành thạo kỹ thuật **lề 0** (ma trận `(m+1)×(n+1)`) để khỏi xử lý riêng hàng/cột đầu.
5. Tránh lỗi: chỉ số âm "im lặng" trong Python, sai dấu bao hàm – loại trừ, lệch chỉ số prefix, tạo ma trận bằng `[[0]*n]*m`.

## 2. Đề bài & ví dụ

Cho ma trận `mat` kích thước `m × n` và số nguyên `k`. Trả về ma trận `answer` cùng kích thước, trong đó `answer[i][j]` là **tổng mọi `mat[r][c]`** thoả:

- `i − k <= r <= i + k`,
- `j − k <= c <= j + k`,
- `(r, c)` là ô hợp lệ trong ma trận.

Nói cách khác: tổng của hình vuông cạnh `2k+1` tâm `(i, j)`, **phần nằm ngoài ma trận bị bỏ**.

**Ví dụ 1:** `mat = [[1,2,3],[4,5,6],[7,8,9]], k = 1` → `[[12,21,16],[27,45,33],[24,39,28]]`

**Ví dụ 2:** cùng `mat`, `k = 2` → `[[45,45,45],[45,45,45],[45,45,45]]` (khối bao trùm cả ma trận).

**Ràng buộc:** `1 <= m, n, k <= 100`, `1 <= mat[i][j] <= 100`.

In [1]:
import random
import time
from itertools import accumulate
from typing import List

MAT = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
examples = [
    ((MAT, 1), [[12, 21, 16], [27, 45, 33], [24, 39, 28]]),
    ((MAT, 2), [[45, 45, 45], [45, 45, 45], [45, 45, 45]]),
]

def show(M, w=4):
    for row in M:
        print("".join(f"{x:{w}d}" for x in row))

for (mat, k), want in examples:
    print(f"k={k}, mong đợi:")
    show(want)

k=1, mong đợi:
  12  21  16
  27  45  33
  24  39  28
k=2, mong đợi:
  45  45  45
  45  45  45
  45  45  45


## 3. Kiến thức nền

### 3.1 Ma trận là list các list; tạo ma trận đúng cách

`[[0] * n for _ in range(m)]` tạo `m` hàng **độc lập**. Còn `[[0] * n] * m` tạo `m` tham chiếu tới **cùng một hàng** — sửa một ô sẽ "lây" sang mọi hàng.

In [2]:
good = [[0] * 3 for _ in range(2)]
bad = [[0] * 3] * 2
good[0][0] = 9
bad[0][0] = 9
print("đúng:", good)
print("sai :", bad, "<- cả hai hàng cùng đổi!")

đúng: [[9, 0, 0], [0, 0, 0]]
sai : [[9, 0, 0], [9, 0, 0]] <- cả hai hàng cùng đổi!


### 3.2 Cắt khối theo biên (clamp)

Khối tâm `(i, j)` bán kính `k`: hàng từ `r1 = max(0, i − k)` đến `r2 = min(m − 1, i + k)`, cột từ `c1 = max(0, j − k)` đến `c2 = min(n − 1, j + k)`.

In [3]:
m, n, k = 3, 3, 1
for i, j in [(0, 0), (1, 1), (2, 1)]:
    r1, r2 = max(0, i - k), min(m - 1, i + k)
    c1, c2 = max(0, j - k), min(n - 1, j + k)
    print(f"tâm ({i},{j}) -> hàng {r1}..{r2}, cột {c1}..{c2}")

tâm (0,0) -> hàng 0..1, cột 0..1
tâm (1,1) -> hàng 0..2, cột 0..2
tâm (2,1) -> hàng 1..2, cột 0..2


### 3.3 Prefix sum 1D

`p[i] = a[0] + … + a[i−1]`, `p[0] = 0`. Tổng đoạn `[l, r]` = `p[r+1] − p[l]`.

In [4]:
a = [4, 5, 6]
p = [0] + list(accumulate(a))
print("p =", p, "| tổng a[1..2] =", p[3] - p[1])

p = [0, 4, 9, 15] | tổng a[1..2] = 11


### 3.4 Prefix sum 2D và bao hàm – loại trừ

Định nghĩa `P[i][j]` = tổng hình chữ nhật từ `(0,0)` tới `(i−1, j−1)` (ma trận `(m+1)×(n+1)`, hàng/cột 0 toàn số 0).

- **Dựng:** `P[i+1][j+1] = mat[i][j] + P[i][j+1] + P[i+1][j] − P[i][j]` (cộng hai hình chữ nhật chồng nhau thì trừ phần chung một lần).
- **Truy vấn** hình chữ nhật hàng `r1..r2`, cột `c1..c2`:

```
S = P[r2+1][c2+1] − P[r1][c2+1] − P[r2+1][c1] + P[r1][c1]
```

Phần trên-trái `P[r1][c1]` bị trừ hai lần nên phải cộng lại một lần.

In [5]:
def prefix2d(mat):
    m, n = len(mat), len(mat[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = mat[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
    return P

def rect(P, r1, c1, r2, c2):
    return P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]

P = prefix2d(MAT)
show(P)
print("tổng hàng 1..2, cột 1..2 =", rect(P, 1, 1, 2, 2), "=", 5 + 6 + 8 + 9)

   0   0   0   0
   0   1   3   6
   0   5  12  21
   0  12  27  45
tổng hàng 1..2, cột 1..2 = 28 = 28


### 3.5 Cửa sổ trượt 1D độ dài thay đổi theo biên

Tổng `a[max(0, j−k) .. min(n−1, j+k)]` cho mọi `j` có thể tính bằng prefix 1D (mỗi `j` `O(1)`). Đây là viên gạch của lời giải "tách chiều".

In [6]:
def window_sums(a, k):
    n = len(a)
    p = [0] + list(accumulate(a))
    return [p[min(n - 1, j + k) + 1] - p[max(0, j - k)] for j in range(n)]

print(window_sums([1, 2, 3], 1), "<- [1+2, 1+2+3, 2+3]")

[3, 6, 5] <- [1+2, 1+2+3, 2+3]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** (`k = 1`), vài ô:

- `answer[0][0]`: khối hàng `0..1`, cột `0..1` ⇒ `1 + 2 + 4 + 5 = 12`.
- `answer[1][1]`: khối hàng `0..2`, cột `0..2` (cả ma trận) ⇒ `45`.
- `answer[2][1]`: hàng `1..2`, cột `0..2` ⇒ `4+5+6+7+8+9 = 39`.

Bằng prefix 2D: `answer[2][1] = P[3][3] − P[1][3] − P[3][0] + P[1][0] = 45 − 6 − 0 + 0 = 39` ✔.

Quan sát: các khối của hai ô cạnh nhau **chồng lấn gần hết** ⇒ cộng lại từ đầu mỗi lần rất lãng phí; tổng tiền tố cho phép trả lời mỗi ô trong `O(1)`.

In [7]:
P = prefix2d(MAT)
for i, j in [(0, 0), (1, 1), (2, 1)]:
    r1, r2, c1, c2 = max(0, i - 1), min(2, i + 1), max(0, j - 1), min(2, j + 1)
    print(f"({i},{j}): P[{r2+1}][{c2+1}]-P[{r1}][{c2+1}]-P[{r2+1}][{c1}]+P[{r1}][{c1}] = "
          f"{P[r2+1][c2+1]}-{P[r1][c2+1]}-{P[r2+1][c1]}+{P[r1][c1]} = {rect(P, r1, c1, r2, c2)}")

(0,0): P[2][2]-P[0][2]-P[2][0]+P[0][0] = 12-0-0+0 = 12
(1,1): P[3][3]-P[0][3]-P[3][0]+P[0][0] = 45-0-0+0 = 45
(2,1): P[3][3]-P[1][3]-P[3][0]+P[1][0] = 45-6-0+0 = 39


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi ô `(i, j)`, cộng trực tiếp mọi ô trong khối đã cắt theo biên.

In [8]:
def block_sum_brute(mat, k):
    m, n = len(mat), len(mat[0])
    ans = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            s = 0
            for r in range(max(0, i - k), min(m - 1, i + k) + 1):
                for c in range(max(0, j - k), min(n - 1, j + k) + 1):
                    s += mat[r][c]
            ans[i][j] = s
    return ans

for (mat, k), want in examples:
    got = block_sum_brute(mat, k)
    print(got, "✔" if got == want else "✘")

[[12, 21, 16], [27, 45, 33], [24, 39, 28]] ✔
[[45, 45, 45], [45, 45, 45], [45, 45, 45]] ✔


**Độ phức tạp:** Time `O(m · n · min(m, 2k+1) · min(n, 2k+1))`. Với `m = n = 100`, `k = 50`: `10^4 · 100 · 100 = 10^8` phép cộng — quá chậm trong Python. Space `O(1)` ngoài kết quả.

**Nút thắt:** các khối liền kề chồng lấn; mỗi ô của `mat` bị cộng lại tới `(2k+1)²` lần.

In [9]:
rng = random.Random(1)
M40 = [[rng.randint(1, 100) for _ in range(40)] for _ in range(40)]
t0 = time.perf_counter()
block_sum_brute(M40, 20)
print(f"brute 40x40, k=20: {time.perf_counter() - t0:.2f}s (100x100, k=50 sẽ chậm hơn ~40 lần)")

brute 40x40, k=20: 0.05s (100x100, k=50 sẽ chậm hơn ~40 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tiền tố theo hàng):** nếu mỗi hàng có prefix 1D thì tổng một đoạn cột trên hàng đó là `O(1)` ⇒ mỗi ô chỉ còn cộng `O(k)` hàng.

**Insight 2 (tiền tố 2D):** tổng mọi hình chữ nhật đều suy ra từ 4 giá trị `P` nhờ **bao hàm – loại trừ** ⇒ mỗi ô `O(1)`, tổng `O(m·n)`.

**Insight 3 (tách chiều):** khối vuông = "tổng theo hàng trong cửa sổ cột" rồi "tổng theo cột trong cửa sổ hàng":
1. `H[i][j]` = tổng `mat[i][c]` với `c ∈ [j−k, j+k]` (cắt biên) — cửa sổ ngang.
2. `answer[i][j]` = tổng `H[r][j]` với `r ∈ [i−k, i+k]` (cắt biên) — cửa sổ dọc.

Vì phép cộng giao hoán, tổng theo khối = tổng các tổng-theo-hàng. Mỗi bước là cửa sổ trượt 1D `O(m·n)`.

**Quy tắc biên:** *nếu* chỉ số khối vượt ra ngoài *thì* cắt về `0` hoặc `m−1`/`n−1` bằng `max`/`min` — **không bao giờ** để chỉ số âm đi vào list Python.

In [10]:
# Minh hoạ Insight 3 trên Ví dụ 1
H = [window_sums(row, 1) for row in MAT]
print("H (cửa sổ ngang):"); show(H)
cols = [window_sums([H[i][j] for i in range(3)], 1) for j in range(3)]
ans = [[cols[j][i] for j in range(3)] for i in range(3)]
print("answer (cửa sổ dọc trên H):"); show(ans)

H (cửa sổ ngang):
   3   6   5
   9  15  11
  15  24  17
answer (cửa sổ dọc trên H):
  12  21  16
  27  45  33
  24  39  28


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Prefix theo từng hàng

Dựng `row_p[i]` = prefix của hàng `i`. Với mỗi ô, duyệt các hàng trong khối và cộng `row_p[r][c2+1] − row_p[r][c1]`.

In [11]:
def block_sum_rows(mat, k):
    m, n = len(mat), len(mat[0])
    row_p = [[0] + list(accumulate(row)) for row in mat]
    ans = [[0] * n for _ in range(m)]
    for i in range(m):
        r1, r2 = max(0, i - k), min(m - 1, i + k)
        for j in range(n):
            c1, c2 = max(0, j - k), min(n - 1, j + k)
            ans[i][j] = sum(row_p[r][c2 + 1] - row_p[r][c1] for r in range(r1, r2 + 1))
    return ans

for (mat, k), want in examples:
    print(block_sum_rows(mat, k) == want)

True
True


Code nộp LeetCode (cách 2):

In [12]:
class SolutionRows:
    def matrixBlockSum(self, mat: List[List[int]], k: int) -> List[List[int]]:
        m, n = len(mat), len(mat[0])
        row_p = [[0] + list(accumulate(row)) for row in mat]
        ans = [[0] * n for _ in range(m)]
        for i in range(m):
            r1, r2 = max(0, i - k), min(m - 1, i + k)
            for j in range(n):
                c1, c2 = max(0, j - k), min(n - 1, j + k)
                ans[i][j] = sum(row_p[r][c2 + 1] - row_p[r][c1] for r in range(r1, r2 + 1))
        return ans

print(SolutionRows().matrixBlockSum(MAT, 1))

[[12, 21, 16], [27, 45, 33], [24, 39, 28]]


**Độ phức tạp cách 2:** Time `O(m · n · min(m, 2k+1))` ≈ `10^6` với giới hạn đề — qua được. Space `O(m·n)`.

### 7.2 Cách 3 — Prefix sum 2D (bao hàm – loại trừ)

1. Dựng `P` kích thước `(m+1) × (n+1)`.
2. Với mỗi ô: cắt biên `r1, r2, c1, c2`, trả `P[r2+1][c2+1] − P[r1][c2+1] − P[r2+1][c1] + P[r1][c1]`.

In [13]:
def block_sum_prefix(mat, k):
    m, n = len(mat), len(mat[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = mat[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
    ans = [[0] * n for _ in range(m)]
    for i in range(m):
        r1, r2 = max(0, i - k), min(m, i + k + 1)      # r2 ở đây là chỉ số trong P (đã +1)
        for j in range(n):
            c1, c2 = max(0, j - k), min(n, j + k + 1)
            ans[i][j] = P[r2][c2] - P[r1][c2] - P[r2][c1] + P[r1][c1]
    return ans

for (mat, k), want in examples:
    print(block_sum_prefix(mat, k) == want)

True
True


Code nộp LeetCode (cách 3):

In [14]:
class SolutionPrefix:
    def matrixBlockSum(self, mat: List[List[int]], k: int) -> List[List[int]]:
        m, n = len(mat), len(mat[0])
        P = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(m):
            row_sum = 0
            for j in range(n):
                row_sum += mat[i][j]                    # tổng hàng i tới cột j
                P[i + 1][j + 1] = P[i][j + 1] + row_sum # = trên + phần hàng hiện tại
        ans = [[0] * n for _ in range(m)]
        for i in range(m):
            r1, r2 = max(0, i - k), min(m, i + k + 1)
            for j in range(n):
                c1, c2 = max(0, j - k), min(n, j + k + 1)
                ans[i][j] = P[r2][c2] - P[r1][c2] - P[r2][c1] + P[r1][c1]
        return ans

print(SolutionPrefix().matrixBlockSum(MAT, 1))

[[12, 21, 16], [27, 45, 33], [24, 39, 28]]


Ghi chú: bản nộp dựng `P` theo công thức tương đương `P[i+1][j+1] = P[i][j+1] + (tổng hàng i từ cột 0 tới j)` — tránh phép trừ, dễ kiểm tra.

**Độ phức tạp cách 3:** Time `O(m · n)`, Space `O(m · n)` cho `P`.

### 7.3 Cách 4 — Tách chiều: cửa sổ ngang rồi cửa sổ dọc

1. Mỗi hàng: `H[i] = window_sums(mat[i], k)`.
2. Mỗi cột của `H`: lấy cửa sổ dọc tương tự ⇒ `answer`.

Không cần công thức 4 số hạng; chỉ dùng hai lần prefix 1D.

In [15]:
def block_sum_separable(mat, k):
    m, n = len(mat), len(mat[0])
    H = []
    for row in mat:                                   # cửa sổ ngang
        p = [0] + list(accumulate(row))
        H.append([p[min(n, j + k + 1)] - p[max(0, j - k)] for j in range(n)])
    ans = [[0] * n for _ in range(m)]
    for j in range(n):                                # cửa sổ dọc trên H
        col = [H[i][j] for i in range(m)]
        p = [0] + list(accumulate(col))
        for i in range(m):
            ans[i][j] = p[min(m, i + k + 1)] - p[max(0, i - k)]
    return ans

for (mat, k), want in examples:
    print(block_sum_separable(mat, k) == want)

True
True


Code nộp LeetCode (cách 4):

In [16]:
class SolutionSeparable:
    def matrixBlockSum(self, mat: List[List[int]], k: int) -> List[List[int]]:
        m, n = len(mat), len(mat[0])
        H = []
        for row in mat:
            p = [0] + list(accumulate(row))
            H.append([p[min(n, j + k + 1)] - p[max(0, j - k)] for j in range(n)])
        ans = [[0] * n for _ in range(m)]
        for j in range(n):
            p = [0] + list(accumulate(H[i][j] for i in range(m)))
            for i in range(m):
                ans[i][j] = p[min(m, i + k + 1)] - p[max(0, i - k)]
        return ans

print(SolutionSeparable().matrixBlockSum(MAT, 2))

[[45, 45, 45], [45, 45, 45], [45, 45, 45]]


**Độ phức tạp cách 4:** Time `O(m · n)`, Space `O(m · n)` cho `H`. Ưu điểm: mở rộng tự nhiên cho bán kính khác nhau theo hàng/cột, và cho 3D (tách 3 lần).

In [17]:
rng = random.Random(2)
M100 = [[rng.randint(1, 100) for _ in range(100)] for _ in range(100)]
for name, f in [("rows", block_sum_rows), ("prefix2d", block_sum_prefix), ("separable", block_sum_separable)]:
    t0 = time.perf_counter()
    f(M100, 50)
    print(f"{name:9s} 100x100, k=50: {time.perf_counter() - t0:.4f}s")

rows      100x100, k=50: 0.0874s
prefix2d  100x100, k=50: 0.0029s
separable 100x100, k=50: 0.0031s


## 8. TRACE từng bước

Trace cách 3 trên Ví dụ 1 (`k = 1`).

**Pha 1 — dựng `P` (4×4):**

| | c=0 | c=1 | c=2 | c=3 |
|---|---|---|---|---|
| r=0 | 0 | 0 | 0 | 0 |
| r=1 | 0 | 1 | 3 | 6 |
| r=2 | 0 | 5 | 12 | 21 |
| r=3 | 0 | 12 | 27 | 45 |

**Pha 2 — một số ô:**

| (i,j) | r1,r2 (trong P) | c1,c2 (trong P) | P[r2][c2] − P[r1][c2] − P[r2][c1] + P[r1][c1] | kết quả |
|---|---|---|---|---|
| (0,0) | 0, 2 | 0, 2 | 12 − 0 − 0 + 0 | 12 |
| (0,2) | 0, 2 | 1, 3 | 21 − 0 − 5 + 0 | 16 |
| (2,2) | 1, 3 | 1, 3 | 45 − 6 − 12 + 1 | 28 |

In [18]:
def trace_prefix(mat, k):
    m, n = len(mat), len(mat[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    print("Pha 1: dựng P")
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = mat[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
            print(f"  P[{i+1}][{j+1}] = mat={mat[i][j]} + trên={P[i][j+1]} + trái={P[i+1][j]} - chéo={P[i][j]} = {P[i+1][j+1]}")
    print("Pha 2: truy vấn từng ô")
    ans = [[0] * n for _ in range(m)]
    for i in range(m):
        r1, r2 = max(0, i - k), min(m, i + k + 1)
        for j in range(n):
            c1, c2 = max(0, j - k), min(n, j + k + 1)
            ans[i][j] = P[r2][c2] - P[r1][c2] - P[r2][c1] + P[r1][c1]
            print(f"  ({i},{j}): {P[r2][c2]} - {P[r1][c2]} - {P[r2][c1]} + {P[r1][c1]} = {ans[i][j]}")
    return ans

res = trace_prefix(MAT, 1)
show(res)

Pha 1: dựng P
  P[1][1] = mat=1 + trên=0 + trái=0 - chéo=0 = 1
  P[1][2] = mat=2 + trên=0 + trái=1 - chéo=0 = 3
  P[1][3] = mat=3 + trên=0 + trái=3 - chéo=0 = 6
  P[2][1] = mat=4 + trên=1 + trái=0 - chéo=0 = 5
  P[2][2] = mat=5 + trên=3 + trái=5 - chéo=1 = 12
  P[2][3] = mat=6 + trên=6 + trái=12 - chéo=3 = 21
  P[3][1] = mat=7 + trên=5 + trái=0 - chéo=0 = 12
  P[3][2] = mat=8 + trên=12 + trái=12 - chéo=5 = 27
  P[3][3] = mat=9 + trên=21 + trái=27 - chéo=12 = 45
Pha 2: truy vấn từng ô
  (0,0): 12 - 0 - 0 + 0 = 12
  (0,1): 21 - 0 - 0 + 0 = 21
  (0,2): 21 - 0 - 5 + 0 = 16
  (1,0): 27 - 0 - 0 + 0 = 27
  (1,1): 45 - 0 - 0 + 0 = 45
  (1,2): 45 - 0 - 12 + 0 = 33
  (2,0): 27 - 3 - 0 + 0 = 24
  (2,1): 45 - 6 - 0 + 0 = 39
  (2,2): 45 - 6 - 12 + 1 = 28
  12  21  16
  27  45  33
  24  39  28


## 9. Code nộp LeetCode

Bản chính thức: prefix sum 2D với lề 0, bao hàm – loại trừ.

In [19]:
class Solution:
    def matrixBlockSum(self, mat: list[list[int]], k: int) -> list[list[int]]:
        m, n = len(mat), len(mat[0])
        # P[i][j] = tổng mat[0..i-1][0..j-1]; hàng/cột 0 là lề 0
        P = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(m):
            for j in range(n):
                P[i + 1][j + 1] = mat[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
        ans = [[0] * n for _ in range(m)]
        for i in range(m):
            r1, r2 = max(0, i - k), min(m, i + k + 1)      # biên hàng (đã cắt, dạng chỉ số P)
            for j in range(n):
                c1, c2 = max(0, j - k), min(n, j + k + 1)  # biên cột
                ans[i][j] = P[r2][c2] - P[r1][c2] - P[r2][c1] + P[r1][c1]
        return ans

sol = Solution()
for (mat, k), want in examples:
    got = sol.matrixBlockSum(mat, k)
    print(got, "✔" if got == want else "✘")

[[12, 21, 16], [27, 45, 33], [24, 39, 28]] ✔
[[45, 45, 45], [45, 45, 45], [45, 45, 45]] ✔


## 10. Vì sao thuật toán đúng

**Dựng `P` đúng:** quy nạp theo `(i, j)`. Hình chữ nhật `R(i, j) = [0..i] × [0..j]` là hợp của `R(i−1, j)` và `R(i, j−1)` cộng thêm ô `(i, j)`; hai hình đó chồng lên nhau đúng ở `R(i−1, j−1)` ⇒ `P[i+1][j+1] = mat[i][j] + P[i][j+1] + P[i+1][j] − P[i][j]`.

**Truy vấn đúng:** gọi `A = R(r2, c2)`. Hình cần tìm = `A` bỏ đi phần trên (`hàng < r1`) và phần trái (`cột < c1`). Hai phần bỏ đi chồng nhau ở góc trên-trái (`hàng < r1`, `cột < c1`) — bị trừ hai lần nên cộng lại một lần ⇒ công thức 4 số hạng.

**Biên đúng:** `r1 = max(0, i−k)` và `r2 = min(m, i+k+1)` (dạng chỉ số `P`, tức hàng cuối là `r2 − 1 = min(m−1, i+k)`) chính là giao của khối với ma trận, đúng như điều kiện "(r, c) hợp lệ" của đề.

**Cách 4:** `Σ_{r} Σ_{c} mat[r][c] = Σ_{r} (Σ_{c} mat[r][c]) = Σ_r H[r][j]` vì miền tổng là tích Descartes `[r1..r2] × [c1..c2]`.

## 11. Phân tích Time / Space

Gọi `m, n` = kích thước (≤ 100), `k` = bán kính (≤ 100), `w = min(m, 2k+1)`, `h = min(n, 2k+1)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(m·n·w·h)` ≤ `10^8` | `O(1)` | chậm với `k` lớn |
| 2. Prefix theo hàng | `O(m·n·w)` ≤ `10^6` | `O(m·n)` | trung gian |
| 3. Prefix 2D | `O(m·n)` | `O(m·n)` | bản nộp chính |
| 4. Tách chiều | `O(m·n)` | `O(m·n)` | mở rộng tốt (bán kính khác nhau, 3D) |

Kết quả `answer` luôn chiếm `O(m·n)`. Các tổng không vượt `100·100·100 = 10^6` — không lo tràn số (Python cũng không giới hạn).

## 12. Bộ test

Ví dụ đề, edge case (ma trận 1×1, 1 hàng, 1 cột, `k` rất lớn, ma trận chữ nhật không vuông), và đối chiếu các cách với brute force trên ma trận ngẫu nhiên (seed cố định, đúng ràng buộc giá trị).

In [20]:
ALL = [block_sum_brute, block_sum_rows, block_sum_prefix, block_sum_separable,
       Solution().matrixBlockSum, SolutionRows().matrixBlockSum,
       SolutionPrefix().matrixBlockSum, SolutionSeparable().matrixBlockSum]
tests = [
    ((MAT, 1), [[12, 21, 16], [27, 45, 33], [24, 39, 28]]),
    ((MAT, 2), [[45] * 3 for _ in range(3)]),
    (([[5]], 3), [[5]]),                                       # 1x1
    (([[1, 2, 3, 4]], 1), [[3, 6, 9, 7]]),                     # 1 hàng
    (([[1], [2], [3], [4]], 1), [[3], [6], [9], [7]]),         # 1 cột
    (([[1, 2], [3, 4], [5, 6]], 1), [[10, 10], [21, 21], [18, 18]]),
    (([[1, 1, 1], [1, 1, 1]], 100), [[6, 6, 6], [6, 6, 6]]),  # k rất lớn
]
for (mat, k), want in tests:
    for f in ALL:
        got = f(mat, k)
        assert got == want, (f, mat, k, got, want)
print("Tất cả", len(tests), "test cố định qua cho", len(ALL), "hàm")

Tất cả 7 test cố định qua cho 8 hàm


In [21]:
rng = random.Random(1314)
for _ in range(300):
    m, n = rng.randint(1, 9), rng.randint(1, 9)
    k = rng.randint(1, 6)
    mat = [[rng.randint(1, 100) for _ in range(n)] for _ in range(m)]
    want = block_sum_brute(mat, k)
    for f in ALL[1:]:
        assert f(mat, k) == want
print("Đối chiếu 300 ma trận ngẫu nhiên: khớp")
t0 = time.perf_counter()
r = Solution().matrixBlockSum(M100, 50)
assert r == block_sum_separable(M100, 50)
print(f"100x100, k=50: Solution khớp cách 4 ({time.perf_counter() - t0:.3f}s gồm cả hai)")

Đối chiếu 300 ma trận ngẫu nhiên: khớp
100x100, k=50: Solution khớp cách 4 (0.010s gồm cả hai)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không cắt biên, để chỉ số âm đi vào list

In [22]:
def wrong_no_clamp(mat, k):
    m, n = len(mat), len(mat[0])
    ans = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            s = 0
            for r in range(i - k, i + k + 1):
                for c in range(j - k, j + k + 1):
                    if r < m and c < n:              # SAI: quên r >= 0, c >= 0
                        s += mat[r][c]               # mat[-1] là hàng CUỐI, không báo lỗi
            ans[i][j] = s
    return ans

print("sai :", wrong_no_clamp(MAT, 1))
print("đúng:", Solution().matrixBlockSum(MAT, 1))

sai : [[45, 45, 33], [45, 45, 33], [39, 39, 28]]
đúng: [[12, 21, 16], [27, 45, 33], [24, 39, 28]]


**Vì sao sai:** Python chấp nhận chỉ số âm (`mat[-1]` = hàng cuối) nên khối "quấn" sang phía đối diện mà không có lỗi nào. Ô `(0,0)` cộng thêm cả hàng cuối và cột cuối. **Sửa:** `range(max(0, i−k), min(m−1, i+k) + 1)` cho cả hàng và cột.

### Lỗi 2 — Sai dấu bao hàm – loại trừ (quên cộng lại góc chéo)

In [23]:
def wrong_sign(mat, k):
    m, n = len(mat), len(mat[0])
    P = prefix2d(mat)
    ans = [[0] * n for _ in range(m)]
    for i in range(m):
        r1, r2 = max(0, i - k), min(m, i + k + 1)
        for j in range(n):
            c1, c2 = max(0, j - k), min(n, j + k + 1)
            ans[i][j] = P[r2][c2] - P[r1][c2] - P[r2][c1]      # SAI: thiếu + P[r1][c1]
    return ans

print("sai :", wrong_sign(MAT, 1))
print("đúng:", Solution().matrixBlockSum(MAT, 1))

sai : [[12, 21, 16], [27, 45, 33], [24, 39, 27]]
đúng: [[12, 21, 16], [27, 45, 33], [24, 39, 28]]


**Vì sao sai:** góc trên-trái bị trừ hai lần (một lần trong "phần trên", một lần trong "phần trái"). Các ô có `r1 > 0` và `c1 > 0` (ví dụ `(2,2)`) bị hụt đúng `P[r1][c1]`. **Sửa:** `+ P[r1][c1]`. Mẹo nhớ: **"lớn − trên − trái + chéo"**.

### Lỗi 3 — Lệch chỉ số giữa `mat` và `P`

In [24]:
def wrong_index(mat, k):
    m, n = len(mat), len(mat[0])
    P = prefix2d(mat)
    ans = [[0] * n for _ in range(m)]
    for i in range(m):
        r1, r2 = max(0, i - k), min(m - 1, i + k)              # chỉ số trong mat
        for j in range(n):
            c1, c2 = max(0, j - k), min(n - 1, j + k)
            ans[i][j] = P[r2][c2] - P[r1][c2] - P[r2][c1] + P[r1][c1]   # SAI: thiếu +1 cho r2, c2
    return ans

print("sai :", wrong_index(MAT, 1))
print("đúng:", Solution().matrixBlockSum(MAT, 1))

sai : [[1, 3, 2], [5, 12, 7], [4, 9, 5]]
đúng: [[12, 21, 16], [27, 45, 33], [24, 39, 28]]


**Vì sao sai:** `P` lệch 1 so với `mat` (`P[i][j]` phủ `mat[0..i−1][0..j−1]`). Dùng `P[r2][c2]` với `r2` là chỉ số `mat` sẽ bỏ mất hàng `r2` và cột `c2`. **Sửa:** dùng `P[r2+1][c2+1]` (hoặc tính `r2 = min(m, i+k+1)` ngay ở dạng chỉ số `P` như bản nộp). Ghi chú rõ "chỉ số này thuộc `mat` hay `P`" khi code.

### Lỗi 4 — Tạo ma trận bằng `[[0] * (n+1)] * (m+1)`

In [25]:
def wrong_alias(mat, k):
    m, n = len(mat), len(mat[0])
    P = [[0] * (n + 1)] * (m + 1)                              # SAI: các hàng dùng chung
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = mat[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
    ans = [[0] * n for _ in range(m)]
    for i in range(m):
        r1, r2 = max(0, i - k), min(m, i + k + 1)
        for j in range(n):
            c1, c2 = max(0, j - k), min(n, j + k + 1)
            ans[i][j] = P[r2][c2] - P[r1][c2] - P[r2][c1] + P[r1][c1]
    return ans

print("sai :", wrong_alias(MAT, 1))
print("đúng:", Solution().matrixBlockSum(MAT, 1))

sai : [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
đúng: [[12, 21, 16], [27, 45, 33], [24, 39, 28]]


**Vì sao sai:** mọi hàng của `P` là **cùng một list**, kể cả "lề 0" ⇒ `P[i][j+1]` và `P[i+1][j+1]` là một ô, công thức tự giẫm lên nhau. **Sửa:** `[[0] * (n + 1) for _ in range(m + 1)]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tổng của mọi hình chữ nhật / khối quanh mỗi ô"** ⇒ prefix sum 2D, công thức "**lớn − trên − trái + chéo**".
- Luôn thêm **lề 0** (kích thước `(m+1)×(n+1)`) để khỏi xử lý riêng hàng/cột 0.
- Viết biên ở dạng **chỉ số `P`** ngay từ đầu: `r1 = max(0, i−k)`, `r2 = min(m, i+k+1)`.
- Khối "tách được" (tích các khoảng) ⇒ có thể làm **từng chiều một** bằng prefix 1D.
- Python: **chỉ số âm không báo lỗi** — luôn `max(0, …)`.
- Kiểm tra nhanh: `k` đủ lớn thì mọi ô bằng tổng cả ma trận.

## 15. Biến thể của bài

1. **LeetCode 304 — Range Sum Query 2D Immutable**: chính là phần truy vấn `rect(P, r1, c1, r2, c2)`.
2. **LeetCode 661 — Image Smoother**: trung bình (làm tròn xuống) khối 3×3 quanh mỗi ô ⇒ cần thêm **số ô** trong khối (bài tập 5).
3. **Bán kính khác nhau theo hàng và cột** (`kr`, `kc`) ⇒ cách 4 xử lý tự nhiên.
4. **LeetCode 1292 — Maximum Side Length of a Square with Sum ≤ Threshold**: prefix 2D + binary search / hai con trỏ.
5. **LeetCode 363 — Max Sum of Rectangle No Larger Than K**: prefix + nén chiều.
6. **Khối 3D** (khối lập phương quanh mỗi ô): tách 3 chiều hoặc prefix 3D (8 số hạng bao hàm – loại trừ).

In [26]:
# Biến thể 3: bán kính kr theo hàng, kc theo cột (dùng tách chiều)
def block_sum_rect(mat, kr, kc):
    m, n = len(mat), len(mat[0])
    H = []
    for row in mat:
        p = [0] + list(accumulate(row))
        H.append([p[min(n, j + kc + 1)] - p[max(0, j - kc)] for j in range(n)])
    ans = [[0] * n for _ in range(m)]
    for j in range(n):
        p = [0] + list(accumulate(H[i][j] for i in range(m)))
        for i in range(m):
            ans[i][j] = p[min(m, i + kr + 1)] - p[max(0, i - kr)]
    return ans

show(block_sum_rect(MAT, 0, 1))      # chỉ cộng theo hàng: cửa sổ ngang
assert block_sum_rect(MAT, 1, 1) == Solution().matrixBlockSum(MAT, 1)

   3   6   5
   9  15  11
  15  24  17


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Cộng trực tiếp từng khối | `O(m·n·w·h)` | `O(1)` | Kiểm chứng |
| 2. Prefix theo hàng | Mỗi hàng prefix 1D, cộng `w` hàng | `O(m·n·w)` | `O(m·n)` | Bước đệm dễ hiểu |
| 3. Prefix 2D | Bao hàm – loại trừ 4 số hạng | `O(m·n)` | `O(m·n)` | Bản nộp chuẩn, truy vấn hình chữ nhật bất kỳ |
| 4. Tách chiều | Cửa sổ ngang rồi dọc | `O(m·n)` | `O(m·n)` | Bán kính khác nhau, nhiều chiều |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().matrixBlockSum([[5]], 3)` và `Solution().matrixBlockSum([[1, 2], [3, 4]], 1)`?

**Bài 2 (tính tay):** Dựng `P` cho `mat = [[2, 1, 3], [4, 0, 1]]` và tính tổng hình chữ nhật hàng `0..1`, cột `1..2` bằng công thức 4 số hạng.

**Bài 3 (điền TODO):** Hoàn thành `build_P(mat)` trả về ma trận prefix `(m+1)×(n+1)`.

**Bài 4 (tự viết lại):** Viết `rect_sum(P, r1, c1, r2, c2)` (chỉ số `mat`, bao gồm hai đầu) không nhìn code.

**Bài 5 (mini challenge):** LeetCode 661 — `image_smoother(img)`: mỗi ô thành **trung bình làm tròn xuống** của các ô trong khối 3×3 quanh nó (chỉ tính ô hợp lệ).

In [27]:
# Bài 3, 4, 5 — khung để tự điền (KHÔNG gọi khi chạy notebook)
def build_P_todo(mat):
    m, n = len(mat), len(mat[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    # TODO: P[i + 1][j + 1] = mat[i][j] + trên + trái - chéo
    return P

def rect_sum_todo(P, r1, c1, r2, c2):
    # TODO: lớn - trên - trái + chéo (nhớ +1 cho r2, c2)
    pass

def image_smoother_todo(img):
    # TODO: tổng khối (k=1) chia cho số ô hợp lệ trong khối
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Ma trận 1×1: khối chỉ có chính nó. (2) Với 2×2 và `k = 1`, mọi khối phủ cả ma trận. (3) Tổng = 10 cho mọi ô.

**Bài 2:** (1) Hàng 1 của `P`: `[0, 2, 3, 6]`. (2) Hàng 2: `[0, 6, 7, 11]`. (3) `P[2][3] − P[0][3] − P[2][1] + P[0][1] = 11 − 0 − 6 + 0 = 5` (= `1 + 3 + 0 + 1`).

**Bài 3:** (1) Hai vòng lặp `i`, `j`. (2) Công thức "ô + trên + trái − chéo". (3) Lưu ý ô `P[i+1][j+1]`.

**Bài 4:** (1) Góc lớn `P[r2+1][c2+1]`. (2) Trừ `P[r1][c2+1]` và `P[r2+1][c1]`. (3) Cộng `P[r1][c1]`.

**Bài 5:** (1) Tổng khối = `matrixBlockSum(img, 1)`. (2) Số ô trong khối = `(r2 − r1 + 1) · (c2 − c1 + 1)` sau khi cắt biên. (3) Dùng `//` để làm tròn xuống.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
assert Solution().matrixBlockSum([[5]], 3) == [[5]]
assert Solution().matrixBlockSum([[1, 2], [3, 4]], 1) == [[10, 10], [10, 10]]

# Bài 3
def build_P(mat):
    m, n = len(mat), len(mat[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            P[i + 1][j + 1] = mat[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
    return P

# Bài 2
P2 = build_P([[2, 1, 3], [4, 0, 1]])
assert P2 == [[0, 0, 0, 0], [0, 2, 3, 6], [0, 6, 7, 11]]
assert P2[2][3] - P2[0][3] - P2[2][1] + P2[0][1] == 5
print("Bài 1–3: OK")

Bài 1–3: OK


In [29]:
# Bài 4
def rect_sum(P, r1, c1, r2, c2):
    return P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]

# Bài 5
def image_smoother(img):
    m, n = len(img), len(img[0])
    P = build_P(img)
    res = [[0] * n for _ in range(m)]
    for i in range(m):
        r1, r2 = max(0, i - 1), min(m - 1, i + 1)
        for j in range(n):
            c1, c2 = max(0, j - 1), min(n - 1, j + 1)
            res[i][j] = rect_sum(P, r1, c1, r2, c2) // ((r2 - r1 + 1) * (c2 - c1 + 1))
    return res

def smoother_brute(img):
    m, n = len(img), len(img[0])
    res = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            cells = [img[r][c] for r in range(i - 1, i + 2) for c in range(j - 1, j + 2)
                     if 0 <= r < m and 0 <= c < n]
            res[i][j] = sum(cells) // len(cells)
    return res

assert image_smoother([[1, 1, 1], [1, 0, 1], [1, 1, 1]]) == [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
assert image_smoother([[100, 200, 100], [200, 50, 200], [100, 200, 100]]) == [[137, 141, 137], [141, 138, 141], [137, 141, 137]]
rng = random.Random(88)
for _ in range(300):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    mat = [[rng.randint(0, 255) for _ in range(n)] for _ in range(m)]
    P = build_P(mat)
    r1 = rng.randrange(m); r2 = rng.randrange(r1, m); c1 = rng.randrange(n); c2 = rng.randrange(c1, n)
    assert rect_sum(P, r1, c1, r2, c2) == sum(mat[r][c] for r in range(r1, r2 + 1) for c in range(c1, c2 + 1))
    assert image_smoother(mat) == smoother_brute(mat)
print("Bài 4–5: OK — mọi assert đều qua")

Bài 4–5: OK — mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template prefix 2D:**

```python
P = [[0] * (n + 1) for _ in range(m + 1)]
for i in range(m):
    for j in range(n):
        P[i+1][j+1] = mat[i][j] + P[i][j+1] + P[i+1][j] - P[i][j]

def rect(r1, c1, r2, c2):        # chỉ số mat, bao gồm hai đầu
    return P[r2+1][c2+1] - P[r1][c2+1] - P[r2+1][c1] + P[r1][c1]
```

**Biên khối tâm `(i, j)` bán kính `k` (dạng chỉ số `P`):**

```python
r1, r2 = max(0, i - k), min(m, i + k + 1)
c1, c2 = max(0, j - k), min(n, j + k + 1)
ans = P[r2][c2] - P[r1][c2] - P[r2][c1] + P[r1][c1]
```

**Checklist trước khi submit:**
- [ ] Ma trận tạo bằng list comprehension (không `[[0]*n]*m`).
- [ ] Cắt biên bằng `max(0, …)` / `min(…)` — không để chỉ số âm.
- [ ] Phân biệt chỉ số `mat` và chỉ số `P` (lệch 1).
- [ ] Công thức "lớn − trên − trái + chéo".
- [ ] Thử `k` lớn: mọi ô bằng tổng cả ma trận.
- [ ] Đúng chữ ký: `matrixBlockSum(self, mat: list[list[int]], k: int) -> list[list[int]]`.